# LeetCode 1750 — Minimum Length of String After Deleting Similar Ends bằng Python

Link đề: https://leetcode.com/problems/minimum-length-of-string-after-deleting-similar-ends/ · Độ khó: **Medium** · Chủ đề: Two Pointers, String

Skeleton LeetCode:

```python
class Solution:
    def minimumLength(self, s: str) -> int:
        ...
```

Lộ trình: **vét cạn mọi cách xóa** (đệ quy có ghi nhớ trên cặp chỉ số) → **mô phỏng bằng cắt chuỗi** `O(n²)` → **hai con trỏ tham lam** `O(n)` / `O(1)` (lời giải nộp) → **nén run-length** rồi hai con trỏ trên các khối.

## 1. Mục tiêu học

1. Mô hình hóa "xóa tiền tố + hậu tố" bằng **cặp chỉ số `[l, r]`** của đoạn còn lại thay vì tạo chuỗi mới.
2. Dùng **hai con trỏ** tiến vào từ hai đầu, mỗi bước "ăn" trọn một **khối ký tự giống nhau** (run).
3. Chứng minh vì sao **xóa tối đa** (cả khối) luôn tối ưu — một lập luận tham lam ngắn gọn.
4. Xử lý chính xác các **điều kiện biên** `l < r` vs `l <= r` (ký tự giữa đơn lẻ không xóa được; khối giữa dài thì xóa hết được).
5. Viết vét cạn bằng `lru_cache` để kiểm chứng tham lam trên chuỗi ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Cho chuỗi `s` chỉ gồm các ký tự `'a'`, `'b'`, `'c'`. Có thể áp dụng thao tác sau **bao nhiêu lần tùy ý** (kể cả 0 lần):

1. Chọn một **tiền tố khác rỗng** của `s` mà mọi ký tự trong đó **giống nhau**.
2. Chọn một **hậu tố khác rỗng** của `s` mà mọi ký tự trong đó **giống nhau**.
3. Tiền tố và hậu tố **không được giao nhau** (không chung chỉ số nào).
4. Ký tự của tiền tố và của hậu tố phải **giống nhau**.
5. Xóa cả tiền tố và hậu tố.

Trả về **độ dài nhỏ nhất** của `s` sau các thao tác.

**Ví dụ 1:** `s = "ca"` → `2` (không xóa được gì).

**Ví dụ 2:** `s = "cabaabac"` → `0`: xóa `c…c` → `"abaaba"`, xóa `a…a` → `"baab"`, xóa `b…b` → `"aa"`, xóa `a…a` → `""`.

**Ví dụ 3:** `s = "aabccabba"` → `3`: xóa `"aa"…"a"` → `"bccabb"`, xóa `"b"…"bb"` → `"cca"`.

**Ràng buộc:** `1 ≤ s.length ≤ 10⁵`; `s` chỉ gồm `'a'`, `'b'`, `'c'`.

In [1]:
import random
import time
from functools import lru_cache
from itertools import groupby
from typing import List

examples = [("ca", 2), ("cabaabac", 0), ("aabccabba", 3)]
for s, want in examples:
    print(repr(s), "-> kỳ vọng", want)

'ca' -> kỳ vọng 2
'cabaabac' -> kỳ vọng 0
'aabccabba' -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1 Chuỗi bất biến và chi phí cắt chuỗi

`s[i:j]` tạo chuỗi **mới** dài `j - i` — tốn `O(j - i)`. Nếu mỗi thao tác đều cắt chuỗi, tổng chi phí có thể là `O(n²)`. Dùng chỉ số `l, r` để "đánh dấu" đoạn còn lại thì không tốn gì.

In [2]:
s = "cabaabac"
t = s[1:-1]
print(repr(t), "| chuỗi mới, s không đổi:", repr(s))
l, r = 1, len(s) - 2
print("cùng đoạn bằng chỉ số:", repr(s[l:r + 1]), "(chỉ in để xem)")

'abaaba' | chuỗi mới, s không đổi: 'cabaabac'
cùng đoạn bằng chỉ số: 'abaaba' (chỉ in để xem)


### 3.2 Đếm độ dài khối ký tự giống nhau ở đầu/cuối

`len(s) - len(s.lstrip(c))` = độ dài tiền tố toàn ký tự `c`. Tương tự `rstrip` cho hậu tố. Tiện cho mô phỏng, nhưng mỗi lần gọi lại tạo chuỗi mới.

In [3]:
s = "aabccabba"
c = s[0]
print("tiền tố toàn", repr(c), "dài", len(s) - len(s.lstrip(c)))
print("hậu tố toàn", repr(s[-1]), "dài", len(s) - len(s.rstrip(s[-1])))

tiền tố toàn 'a' dài 2
hậu tố toàn 'a' dài 1


### 3.3 Hai con trỏ `l`, `r`

Mẫu "đi vào từ hai đầu": `l` bắt đầu ở 0, `r` ở `n-1`; mỗi bước tăng `l` hoặc giảm `r`. Đoạn còn lại là `s[l..r]`, độ dài `r - l + 1` (có thể bằng 0 khi `l = r + 1`).

In [4]:
s = "abcba"
l, r = 0, len(s) - 1
while l < r and s[l] == s[r]:
    print(f"l={l} r={r}: s[l]={s[l]!r} == s[r]={s[r]!r}")
    l += 1; r -= 1
print("dừng tại l =", l, "r =", r, "-> còn", r - l + 1, "ký tự")

l=0 r=4: s[l]='a' == s[r]='a'
l=1 r=3: s[l]='b' == s[r]='b'
dừng tại l = 2 r = 2 -> còn 1 ký tự


### 3.4 `itertools.groupby` để nén run-length

`groupby("aabccabba")` gom các ký tự liên tiếp giống nhau: `[('a',2),('b',1),('c',2),('a',1),('b',2),('a',1)]`.

In [5]:
runs = [(ch, len(list(g))) for ch, g in groupby("aabccabba")]
print(runs)

[('a', 2), ('b', 1), ('c', 2), ('a', 1), ('b', 2), ('a', 1)]


### 3.5 `functools.lru_cache` cho vét cạn có ghi nhớ

Trạng thái của bài là cặp `(l, r)` — chỉ có `O(n²)` cặp. Đệ quy thử mọi thao tác, ghi nhớ kết quả mỗi cặp.

In [6]:
@lru_cache(maxsize=None)
def count_paths(i, j):
    if i == 0 or j == 0:
        return 1
    return count_paths(i - 1, j) + count_paths(i, j - 1)

print(count_paths(10, 10), count_paths.cache_info().currsize, "trạng thái")

184756 120 trạng thái


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 3 `s = "aabccabba"`, tính bằng tay với hai con trỏ:

```
a a b c c a b b a
l               r      s[l] = s[r] = 'a'  -> ăn hết khối 'a' đầu ("aa") và cuối ("a")
    l         r        s[l] = 'b' = s[r]  -> ăn "b" đầu và "bb" cuối
      l   r            s[l] = 'c' ≠ 'a' = s[r] -> DỪNG
```

Còn `"cca"` dài 3. ✔

**Vì sao ăn cả khối?** Nếu chỉ xóa một phần khối `'a'` đầu, chuỗi vẫn bắt đầu bằng `'a'`; mọi thao tác tiếp theo **bắt buộc** dùng ký tự `'a'` ở đầu — vậy gộp chúng làm một lần cũng vậy. Xóa nhiều hơn không bao giờ làm mất cơ hội.

**Trường hợp ký tự giữa:** `"aba"` → xóa `a…a` còn `"b"`. Tiền tố `"b"` và hậu tố `"b"` **trùng** chỉ số ⇒ không được xóa. Đáp án 1. Nhưng `"abba"` → còn `"bb"` → tiền tố `"b"`, hậu tố `"b"` khác chỉ số ⇒ xóa được, đáp án 0.

In [7]:
def show_steps(s):
    l, r = 0, len(s) - 1
    while l < r and s[l] == s[r]:
        c = s[l]
        l0, r0 = l, r
        while l <= r and s[l] == c: l += 1
        while l <= r and s[r] == c: r -= 1
        print(f"ăn {c!r}: tiền tố s[{l0}:{l}]={s[l0:l]!r}, hậu tố s[{r+1}:{r0+1}]={s[r+1:r0+1]!r} -> còn {s[l:r+1]!r}")
    return r - l + 1

for s in ["aabccabba", "aba", "abba"]:
    print(repr(s), "=> độ dài", show_steps(s)); print()

ăn 'a': tiền tố s[0:2]='aa', hậu tố s[8:9]='a' -> còn 'bccabb'
ăn 'b': tiền tố s[2:3]='b', hậu tố s[6:8]='bb' -> còn 'cca'
'aabccabba' => độ dài 3

ăn 'a': tiền tố s[0:1]='a', hậu tố s[2:3]='a' -> còn 'b'
'aba' => độ dài 1

ăn 'a': tiền tố s[0:1]='a', hậu tố s[3:4]='a' -> còn 'bb'
ăn 'b': tiền tố s[1:3]='bb', hậu tố s[3:3]='' -> còn ''
'abba' => độ dài 0



## 5. Cách 1 — Brute Force / baseline: thử **mọi** thao tác (đệ quy có ghi nhớ)

**Ý tưởng:** trạng thái là đoạn còn lại `s[l..r]`. Nếu `s[l] == s[r]` và `l < r`, ta có thể chọn tiền tố dài `p` (toàn `s[l]`) và hậu tố dài `q` (toàn `s[r]`) bất kỳ, miễn không giao nhau. Thử hết mọi `(p, q)`, lấy kết quả nhỏ nhất; cũng có thể dừng (giữ nguyên độ dài).

Không giả định tham lam ⇒ là chuẩn đối chiếu.

In [8]:
def min_length_brute(s):
    n = len(s)

    @lru_cache(maxsize=None)
    def go(l, r):
        best = r - l + 1                       # dừng ở đây
        if l < r and s[l] == s[r]:
            c = s[l]
            p_max = 0
            while l + p_max <= r and s[l + p_max] == c:
                p_max += 1
            q_max = 0
            while r - q_max >= l and s[r - q_max] == c:
                q_max += 1
            for p in range(1, p_max + 1):
                for q in range(1, q_max + 1):
                    if l + p - 1 < r - q + 1:  # không giao nhau
                        best = min(best, go(l + p, r - q))
        return best

    res = go(0, n - 1)
    go.cache_clear()
    return res

for s, want in examples:
    print(repr(s), "->", min_length_brute(s), "| kỳ vọng", want)

'ca' -> 2 | kỳ vọng 2
'cabaabac' -> 0 | kỳ vọng 0
'aabccabba' -> 3 | kỳ vọng 3


**Độ phức tạp:** `O(n²)` trạng thái, mỗi trạng thái thử tới `O(n²)` cặp `(p, q)` ⇒ worst `O(n⁴)` (chuỗi toàn một ký tự). Bộ nhớ `O(n²)`, đệ quy sâu tới `O(n)`. Chỉ dùng cho `n ≤ 30`.

**Nút thắt:** ta thử cả những cách xóa "một phần khối" — mà trực giác ở mục 4 cho thấy chúng không bao giờ tốt hơn xóa cả khối.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (tham lam):** tại đoạn `s[l..r]` với `s[l] == s[r] = c` và `l < r`, **xóa toàn bộ khối `c` ở đầu và toàn bộ khối `c` ở cuối** là tối ưu.

**Quy tắc "nếu… thì…":**
- *Nếu* `l < r` và `s[l] == s[r]` *thì* ăn hết `c` từ trái (`l` tăng khi `s[l] == c`) và từ phải (`r` giảm khi `s[r] == c`), với điều kiện con trỏ chưa vượt nhau (`l <= r`).
- *Nếu* `l == r` (còn đúng 1 ký tự) *thì* dừng — không chọn được tiền tố và hậu tố rời nhau.
- *Nếu* `s[l] != s[r]` *thì* dừng — điều kiện 4 bị vi phạm.

**Chi tiết tinh tế:** điều kiện vòng ngoài là `l < r` (cần ít nhất 2 ký tự), nhưng vòng trong dùng `l <= r` để khi cả đoạn còn lại toàn `c` (ví dụ `"aaa"`), con trỏ `l` ăn hết được — tiền tố `"a"` + hậu tố `"aa"` là hợp lệ vì có ≥ 2 ký tự.

**Invariant:** sau mỗi vòng, `s[l..r]` là chuỗi đạt được bằng một dãy thao tác hợp lệ, và độ dài tối ưu từ `s[0..n-1]` bằng độ dài tối ưu từ `s[l..r]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng tham lam bằng cắt chuỗi (`O(n²)` worst)

In [9]:
def min_length_slicing(s):
    while len(s) >= 2 and s[0] == s[-1]:
        c = s[0]
        s = s.lstrip(c)          # bỏ khối c đầu (tạo chuỗi mới)
        s = s.rstrip(c)          # bỏ khối c cuối
    return len(s)

for s, want in examples:
    print(repr(s), min_length_slicing(s), want)

'ca' 2 2
'cabaabac' 0 0
'aabccabba' 3 3


Lưu ý `lstrip` có thể ăn **cả** chuỗi khi chuỗi toàn `c` — vẫn đúng vì độ dài ≥ 2 nên chia được thành tiền tố + hậu tố.

**Độ phức tạp:** mỗi vòng tạo chuỗi mới `O(n)`; số vòng có thể tới `~n/2` (ví dụ `"abcabc…cba"` đối xứng, mỗi khối dài 1) ⇒ `O(n²)` worst. Bộ nhớ `O(n)`. Nút thắt: **sao chép chuỗi** — thay bằng hai chỉ số.

### 7.2 Cách 3 — Hai con trỏ tham lam (`O(n)` thời gian, `O(1)` bộ nhớ)

In [10]:
class SolutionTwoPointers:
    def minimumLength(self, s: str) -> int:
        l, r = 0, len(s) - 1
        while l < r and s[l] == s[r]:
            c = s[l]
            while l <= r and s[l] == c:
                l += 1
            while l <= r and s[r] == c:
                r -= 1
        return r - l + 1

for s, want in examples:
    print(repr(s), SolutionTwoPointers().minimumLength(s), want)

'ca' 2 2
'cabaabac' 0 0
'aabccabba' 3 3


**Độ phức tạp:** mỗi lần `l += 1` hoặc `r -= 1` thu hẹp đoạn đi 1, tổng cộng không quá `n` lần ⇒ `O(n)`. Bộ nhớ `O(1)`.

Kết quả `r - l + 1` không bao giờ âm: vòng trong chỉ cho `l` vượt `r` đúng 1 (`l = r + 1`) ⇒ độ dài 0.

### 7.3 Cách 4 — Nén run-length, hai con trỏ trên các khối

Nén `s` thành danh sách khối `(ký tự, độ dài)`. Hai con trỏ `i`, `j` trên danh sách khối:
- *Nếu* `i < j` và hai khối cùng ký tự ⇒ xóa cả hai khối (`i += 1`, `j -= 1`).
- *Nếu* `i == j` ⇒ còn một khối duy nhất: độ dài ≥ 2 thì xóa hết được (đáp án 0), dài 1 thì còn 1.
- Ngược lại dừng, đáp án = tổng độ dài các khối `i..j`.

Chú ý: hai khối kề cạnh nhau trong RLE luôn khác ký tự, nên khi `i + 1 == j` và cùng ký tự thì không xảy ra — an toàn.

In [11]:
class SolutionRLE:
    def minimumLength(self, s: str) -> int:
        runs = [(ch, len(list(g))) for ch, g in groupby(s)]
        i, j = 0, len(runs) - 1
        while i < j and runs[i][0] == runs[j][0]:
            i += 1
            j -= 1
        if i > j:
            return 0
        if i == j:
            return 0 if runs[i][1] >= 2 else 1
        return sum(length for _, length in runs[i:j + 1])

for s, want in examples:
    print(repr(s), SolutionRLE().minimumLength(s), want)

'ca' 2 2
'cabaabac' 0 0
'aabccabba' 3 3


**Độ phức tạp:** `O(n)` để nén, `O(k)` duyệt `k` khối ⇒ `O(n)`; bộ nhớ `O(k)` ≤ `O(n)`. Cách này tách bạch **cấu trúc** (các khối) khỏi **chi tiết** (chỉ số ký tự) — dễ lập luận hơn.

Câu hỏi: khi `i == j` mà khối giữa có độ dài ≥ 2, vì sao xóa hết được? Vì cả đoạn còn lại toàn một ký tự `c` dài `L ≥ 2`: chọn tiền tố dài 1 và hậu tố dài `L − 1`, rời nhau ⇒ còn 0.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 2: `s = "cabaabac"` (chỉ số 0..7).

| vòng | `l, r` đầu vòng | `c` | sau ăn trái `l` | sau ăn phải `r` | đoạn còn lại |
|---|---|---|---|---|---|
| 1 | 0, 7 | c | 1 | 6 | `"abaaba"` |
| 2 | 1, 6 | a | 2 | 5 | `"baab"` |
| 3 | 2, 5 | b | 3 | 4 | `"aa"` |
| 4 | 3, 4 | a | 5 (ăn cả `s[3], s[4]`) | 4 (vòng phải không chạy vì `l > r`) | `""` |

Kết thúc `l = 5, r = 4` ⇒ `r − l + 1 = 0`. Ở vòng 4, vòng trái ăn **cả hai** ký tự `'a'` — đó là lý do vòng trong cần `l <= r`.

In [12]:
def trace_two_pointers(s):
    l, r = 0, len(s) - 1
    step = 0
    while l < r and s[l] == s[r]:
        step += 1
        c = s[l]
        print(f"vòng {step}: l={l}, r={r}, c={c!r}")
        while l <= r and s[l] == c:
            print(f"    ăn trái s[{l}]={s[l]!r}")
            l += 1
        while l <= r and s[r] == c:
            print(f"    ăn phải s[{r}]={s[r]!r}")
            r -= 1
        print(f"    -> l={l}, r={r}, còn {s[l:r+1]!r}")
    reason = "l >= r" if l >= r else f"s[l]={s[l]!r} != s[r]={s[r]!r}"
    print(f"dừng vì {reason}; độ dài = {r - l + 1}")
    return r - l + 1

trace_two_pointers("cabaabac")

vòng 1: l=0, r=7, c='c'
    ăn trái s[0]='c'
    ăn phải s[7]='c'
    -> l=1, r=6, còn 'abaaba'
vòng 2: l=1, r=6, c='a'
    ăn trái s[1]='a'
    ăn phải s[6]='a'
    -> l=2, r=5, còn 'baab'
vòng 3: l=2, r=5, c='b'
    ăn trái s[2]='b'
    ăn phải s[5]='b'
    -> l=3, r=4, còn 'aa'
vòng 4: l=3, r=4, c='a'
    ăn trái s[3]='a'
    ăn trái s[4]='a'
    -> l=5, r=4, còn ''
dừng vì l >= r; độ dài = 0


0

In [13]:
trace_two_pointers("aabccabba")

vòng 1: l=0, r=8, c='a'
    ăn trái s[0]='a'
    ăn trái s[1]='a'
    ăn phải s[8]='a'
    -> l=2, r=7, còn 'bccabb'
vòng 2: l=2, r=7, c='b'
    ăn trái s[2]='b'
    ăn phải s[7]='b'
    ăn phải s[6]='b'
    -> l=3, r=5, còn 'cca'
dừng vì s[l]='c' != s[r]='a'; độ dài = 3


3

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def minimumLength(self, s: str) -> int:
        l, r = 0, len(s) - 1
        while l < r and s[l] == s[r]:        # cần >= 2 ký tự và hai đầu giống nhau
            c = s[l]
            while l <= r and s[l] == c:      # ăn trọn khối c bên trái
                l += 1
            while l <= r and s[r] == c:      # ăn trọn khối c bên phải
                r -= 1
        return r - l + 1

for s, want in examples:
    assert Solution().minimumLength(s) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `opt(l, r)` là độ dài nhỏ nhất đạt được từ đoạn `s[l..r]`.

**Bổ đề (xóa tối đa không thua):** giả sử `l < r`, `s[l] = s[r] = c`, khối `c` đầu là `s[l..a-1]`, khối `c` cuối là `s[b+1..r]` (đã chặn để không vượt nhau). Ta chứng minh `opt(l, r) = opt(a, b)`.

- *`≤`:* xóa tiền tố `s[l..a-1]` và hậu tố `s[b+1..r]` là **một** thao tác hợp lệ (cả hai toàn `c`, khác rỗng, rời nhau — vì `l < r` nên chia được), dẫn tới `s[a..b]`.
- *`≥`:* xét một dãy thao tác tối ưu bất kỳ từ `s[l..r]`. Nếu nó không làm gì, độ dài `r − l + 1 ≥` mọi thứ khác. Nếu có, thao tác đầu tiên bắt buộc dùng ký tự `c` và chỉ xóa **một phần** các khối `c` hai đầu (tiền tố toàn `c` không vượt quá khối `c` đầu). Phần `c` còn sót lại ở đầu (nếu có) chỉ có thể bị xóa bởi các thao tác **cũng dùng ký tự `c`** ở cả hai đầu; ta gộp chúng lại thành một lần xóa tối đa mà không làm mất thao tác nào về sau. Lập luận chi tiết hơn: mọi chuỗi đạt được từ `s[l..r]` qua ≥ 1 thao tác và **không còn `c` ở hai đầu** đều đạt được từ `s[a..b]`; còn chuỗi kết thúc vẫn có `c` ở một đầu thì dài hơn hoặc bằng chuỗi tương ứng sau khi xóa nốt.

**Dừng:** nếu `l == r` không có cặp tiền tố/hậu tố rời nhau; nếu `s[l] != s[r]` vi phạm điều kiện 4 ⇒ `opt = r − l + 1`.

Ngoài chứng minh, mục 12 đối chiếu tham lam với vét cạn trên hàng nghìn chuỗi ngẫu nhiên.

## 11. Phân tích Time / Space

Gọi `n = len(s)`, `k` = số khối run-length.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn ghi nhớ | `O(n⁴)` worst | `O(n²)` |
| 2. Cắt chuỗi | `O(n²)` worst, `O(n)` khi ít vòng | `O(n)` |
| 3. Hai con trỏ | `O(n)` | `O(1)` |
| 4. RLE | `O(n)` | `O(k)` |

Đo trên input worst cho Cách 2 (chuỗi đối xứng, khối dài 1) với `n = 10⁵`:

In [15]:
half = "".join("abc"[i % 3] for i in range(50000))
worst = half + half[::-1]
for name, f in [("Cách 2 cắt chuỗi", min_length_slicing),
                ("Cách 3 hai con trỏ", SolutionTwoPointers().minimumLength),
                ("Cách 4 RLE", SolutionRLE().minimumLength)]:
    t = time.perf_counter()
    r = f(worst)
    print(f"{name}: {r} trong {1000*(time.perf_counter()-t):.1f} ms")

Cách 2 cắt chuỗi: 0 trong 115.3 ms
Cách 3 hai con trỏ: 0 trong 6.6 ms


Cách 4 RLE: 0 trong 19.2 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [16]:
sols = [Solution(), SolutionTwoPointers(), SolutionRLE()]
edge = examples + [
    ("a", 1),             # 1 ký tự: không xóa được
    ("aa", 0),            # 2 ký tự giống
    ("ab", 2),
    ("aaa", 0),           # toàn một ký tự dài >= 2
    ("aba", 1),           # ký tự giữa đơn lẻ
    ("abba", 0),
    ("abbba", 0),         # khối giữa dài 3
    ("abcba", 1),
    ("b" * 1000, 0),      # dài, toàn một ký tự
    ("abcabc", 6),
    ("aabbaabbaa", 0),
    ("b" * 40 + "aaaaab", 0),
]
for s, want in edge:
    for sol in sols:
        assert sol.minimumLength(s) == want, (s, sol)
    assert min_length_slicing(s) == want
    if len(s) <= 30:
        assert min_length_brute(s) == want
    print("OK", repr(s[:20]) + ("…" if len(s) > 20 else ""), "->", want)

OK 'ca' -> 2
OK 'cabaabac' -> 0
OK 'aabccabba' -> 3
OK 'a' -> 1
OK 'aa' -> 0
OK 'ab' -> 2
OK 'aaa' -> 0
OK 'aba' -> 1
OK 'abba' -> 0
OK 'abbba' -> 0
OK 'abcba' -> 1
OK 'bbbbbbbbbbbbbbbbbbbb'… -> 0
OK 'abcabc' -> 6
OK 'aabbaabbaa' -> 0
OK 'bbbbbbbbbbbbbbbbbbbb'… -> 0


### 12.2 Ngẫu nhiên có seed, đối chiếu vét cạn

Sinh chuỗi trên bảng chữ `{a, b, c}` hoặc `{a, b}` (dễ có nhiều cặp đầu-cuối giống nhau), và cả chuỗi "đối xứng có nhiễu" để thao tác chạy nhiều vòng.

In [17]:
rng = random.Random(1750)
cnt = 0
for _ in range(1500):
    alpha = rng.choice(["ab", "abc", "a", "aab"])
    n = rng.randint(1, 14)
    s = "".join(rng.choice(alpha) for _ in range(n))
    if rng.random() < 0.4:
        s = s + s[::-1][rng.randint(0, 1):]          # gần đối xứng
    s = s[:20]
    want = min_length_brute(s)
    for sol in sols:
        assert sol.minimumLength(s) == want, s
    assert min_length_slicing(s) == want
    cnt += 1
print("đã đối chiếu", cnt, "chuỗi với vét cạn")

đã đối chiếu 1500 chuỗi với vét cạn


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Vòng ngoài dùng `l <= r` (cho phép xóa ký tự đơn ở giữa)

In [18]:
def wrong_outer_le(s):
    l, r = 0, len(s) - 1
    while l <= r and s[l] == s[r]:          # SAI: khi l == r vẫn "xóa"
        c = s[l]
        while l <= r and s[l] == c: l += 1
        while l <= r and s[r] == c: r -= 1
    return r - l + 1

for s in ["a", "aba", "abcba"]:
    print(repr(s), "sai:", wrong_outer_le(s), "| đúng:", Solution().minimumLength(s))

'a' sai: 0 | đúng: 1
'aba' sai: 0 | đúng: 1
'abcba' sai: 0 | đúng: 1


**Vì sao sai:** với 1 ký tự, tiền tố và hậu tố khác rỗng **bắt buộc** trùng chỉ số đó ⇒ vi phạm điều kiện 3. **Sửa:** vòng ngoài `l < r`.

### Lỗi 2 — Vòng trong thiếu điều kiện chặn con trỏ

In [19]:
def wrong_no_guard(s):
    l, r = 0, len(s) - 1
    while l < r and s[l] == s[r]:
        c = s[l]
        while s[l] == c: l += 1             # SAI: có thể chạy quá cuối chuỗi
        while s[r] == c: r -= 1
    return r - l + 1

for s in ["aa", "aaa"]:
    try:
        print(repr(s), "->", wrong_no_guard(s))
    except IndexError as e:
        print(repr(s), "-> IndexError:", e)
print("đúng:", Solution().minimumLength("aa"), Solution().minimumLength("aaa"))

'aa' -> IndexError: string index out of range
'aaa' -> IndexError: string index out of range
đúng: 0 0


**Vì sao sai:** với chuỗi toàn `'a'`, `l` tăng tới `len(s)` rồi `s[l]` vượt chỉ số. (Thú vị: `r` giảm xuống âm thì Python **không** báo lỗi mà lặng lẽ đọc `s[-1]` — một lỗi im lặng còn nguy hiểm hơn.) **Sửa:** thêm `l <= r` vào cả hai vòng trong.

### Lỗi 3 — Mỗi bước chỉ xóa một ký tự ở mỗi đầu

In [20]:
def wrong_one_char(s):
    l, r = 0, len(s) - 1
    while l < r and s[l] == s[r]:
        l += 1; r -= 1                      # SAI: không ăn hết khối
    return r - l + 1

for s in ["aabccabba", "aab" + "a"]:
    print(repr(s), "sai:", wrong_one_char(s), "| đúng:", Solution().minimumLength(s))

'aabccabba' sai: 7 | đúng: 3
'aaba' sai: 2 | đúng: 1


**Vì sao sai:** `"aabccabba"`: sau khi xóa `a…a`, đoạn còn `"abccabb"` có đầu `'a'` ≠ cuối `'b'` ⇒ dừng ở 7. Nhưng tiền tố được phép dài **nhiều ký tự**: xóa `"aa"…"a"` cùng lúc thì tiếp tục được. Đây là phiên bản "kiểm tra palindrome" — **không phải** bài này. **Sửa:** ăn trọn khối.

### Lỗi 4 — Vòng trong dùng `l < r` thay vì `l <= r`

In [21]:
def wrong_inner_lt(s):
    l, r = 0, len(s) - 1
    while l < r and s[l] == s[r]:
        c = s[l]
        while l < r and s[l] == c: l += 1   # SAI: giữ lại 1 ký tự cuối của khối
        while l < r and s[r] == c: r -= 1
    return r - l + 1

for s in ["aaa", "abbba", "aa"]:
    print(repr(s), "sai:", wrong_inner_lt(s), "| đúng:", Solution().minimumLength(s))

'aaa' sai: 1 | đúng: 0
'abbba' sai: 1 | đúng: 0
'aa' sai: 1 | đúng: 0


**Vì sao sai:** khi đoạn còn lại toàn `c` dài `≥ 2`, nó xóa được **hết** (tiền tố 1 ký tự + hậu tố phần còn lại). Dùng `l < r` khiến con trỏ dừng ở `l == r` và để sót 1 ký tự. **Sửa:** vòng trong `l <= r` (vòng ngoài mới là `l < r`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Xóa/so khớp từ hai đầu"** ⇒ hai con trỏ `l`, `r` + đoạn `s[l..r]`, đừng cắt chuỗi.
- **Xóa "khối ký tự giống nhau"** ⇒ vòng `while` trong để ăn trọn khối, hoặc nén run-length trước.
- Phân biệt hai điều kiện: vòng ngoài là **điều kiện được phép thao tác** (`l < r`), vòng trong là **điều kiện không vượt biên** (`l <= r`).
- Tham lam "làm tối đa mỗi bước" thường đúng khi phần làm dở **buộc** bước sau phải làm tiếp cùng loại.
- Test nhanh: `"a"`, `"aa"`, `"aaa"`, `"aba"`, `"abba"`.

## 15. Biến thể của bài

1. **Chỉ xóa đúng một ký tự mỗi đầu** (kiểm tra palindrome / LeetCode 125, 680 *Valid Palindrome II*).
2. **Trả về chính chuỗi còn lại** thay vì độ dài: `s[l:r+1]`.
3. **Nhiều truy vấn trên chuỗi con** `s[L..R]`: chạy hai con trỏ trên đoạn (Bài tập 5); có thể tăng tốc bằng RLE + nhảy khối.
4. **Bảng chữ cái lớn hơn** (`a..z`): thuật toán không đổi.
5. **Cho phép ký tự tiền tố và hậu tố khác nhau, chi phí khác nhau…** ⇒ thường chuyển thành DP trên đoạn `(l, r)` như Cách 1.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn ghi nhớ | thử mọi `(p, q)` trên trạng thái `(l, r)` | `O(n⁴)` | `O(n²)` | kiểm chứng tham lam |
| 2. Cắt chuỗi | `lstrip`/`rstrip` lặp | `O(n²)` | `O(n)` | viết nhanh khi `n` nhỏ |
| 3. Hai con trỏ | ăn trọn khối hai đầu | `O(n)` | `O(1)` | **nộp bài** |
| 4. RLE | hai con trỏ trên khối | `O(n)` | `O(k)` | dễ chứng minh, nhiều truy vấn |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `minimumLength` của `"aba"`, `"abba"`, `"abccba"` và `"abcab"`?

**Bài 2 (tính tay):** Chạy hai con trỏ trên `"bbaabbaab"`: ghi `l, r` sau mỗi vòng và đáp án.

**Bài 3 (điền TODO):** Viết `rle(s)` trả về list `(ký tự, độ dài)` **không dùng** `groupby`.

**Bài 4 (tự viết lại):** Viết `remaining(s)` trả về **chuỗi** còn lại sau tham lam (không chỉ độ dài).

**Bài 5 (mini challenge):** Cho `s` và danh sách truy vấn `[L, R]` (chỉ số đóng). Với mỗi truy vấn trả về đáp án của bài cho chuỗi con `s[L..R]`. Viết `answer_queries(s, queries)` không tạo chuỗi con (dùng chỉ số).

In [22]:
def rle(s):
    # TODO: không dùng groupby
    pass

def remaining(s):
    # TODO: trả về chuỗi còn lại
    pass

def answer_queries(s, queries):
    # TODO: hai con trỏ trên đoạn [L, R]
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Chạy tay hai con trỏ | `"aba"` còn `"b"` | `"abcab"`: `a` ≠ `b` ngay từ đầu |
| 2 | `n = 9`, `l=0, r=8` | Vòng 1 `c='b'`: `l` → 2, `r` → 7 | Tiếp `'a'`… cuối cùng `l > r` |
| 3 | Duyệt và so với ký tự trước | Khi khác thì chốt khối cũ | Đừng quên chốt khối cuối sau vòng lặp |
| 4 | Giữ nguyên Cách 3 | Trả `s[l:r+1]` | Khi `l = r + 1` lát cắt là `""` |
| 5 | Thay `0, n-1` bằng `L, R` | Vòng ngoài `l < r`, vòng trong `l <= r` | Đối chiếu với `Solution().minimumLength(s[L:R+1])` |

## 19. Đáp án tham khảo

In [23]:
for s in ["aba", "abba", "abccba", "abcab"]:
    print("Bài 1:", repr(s), "->", Solution().minimumLength(s))
assert [Solution().minimumLength(s) for s in ["aba", "abba", "abccba", "abcab"]] == [1, 0, 0, 5]
print("Bài 2:")
assert trace_two_pointers("bbaabbaab") == 0

Bài 1: 'aba' -> 1
Bài 1: 'abba' -> 0
Bài 1: 'abccba' -> 0
Bài 1: 'abcab' -> 5
Bài 2:
vòng 1: l=0, r=8, c='b'
    ăn trái s[0]='b'
    ăn trái s[1]='b'
    ăn phải s[8]='b'
    -> l=2, r=7, còn 'aabbaa'
vòng 2: l=2, r=7, c='a'
    ăn trái s[2]='a'
    ăn trái s[3]='a'
    ăn phải s[7]='a'
    ăn phải s[6]='a'
    -> l=4, r=5, còn 'bb'
vòng 3: l=4, r=5, c='b'
    ăn trái s[4]='b'
    ăn trái s[5]='b'
    -> l=6, r=5, còn ''
dừng vì l >= r; độ dài = 0


In [24]:
def rle(s):
    out = []
    for ch in s:
        if out and out[-1][0] == ch:
            out[-1] = (ch, out[-1][1] + 1)
        else:
            out.append((ch, 1))
    return out

def remaining(s):
    l, r = 0, len(s) - 1
    while l < r and s[l] == s[r]:
        c = s[l]
        while l <= r and s[l] == c: l += 1
        while l <= r and s[r] == c: r -= 1
    return s[l:r + 1]

rng = random.Random(3)
for _ in range(500):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 15)))
    assert rle(s) == [(ch, len(list(g))) for ch, g in groupby(s)]
    rem = remaining(s)
    assert len(rem) == Solution().minimumLength(s) and rem in s
print("Bài 3, 4: OK", rle("aabccabba"), repr(remaining("aabccabba")))

Bài 3, 4: OK [('a', 2), ('b', 1), ('c', 2), ('a', 1), ('b', 2), ('a', 1)] 'cca'


In [25]:
def answer_queries(s, queries):
    res = []
    for L, R in queries:
        l, r = L, R
        while l < r and s[l] == s[r]:
            c = s[l]
            while l <= r and s[l] == c: l += 1
            while l <= r and s[r] == c: r -= 1
        res.append(r - l + 1)
    return res

rng = random.Random(9)
for _ in range(100):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 16)))
    qs = []
    for _ in range(5):
        L = rng.randrange(len(s)); R = rng.randrange(L, len(s)); qs.append([L, R])
    assert answer_queries(s, qs) == [min_length_brute(s[L:R + 1]) for L, R in qs]
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", answer_queries("cabaabac", [[0, 7], [1, 3], [2, 5]]))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK [0, 1, 0]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
l, r = 0, len(s) - 1
while l < r and s[l] == s[r]:          # được phép thao tác: >= 2 ký tự, hai đầu giống
    c = s[l]
    while l <= r and s[l] == c: l += 1 # ăn trọn khối trái (không vượt biên)
    while l <= r and s[r] == c: r -= 1 # ăn trọn khối phải
return r - l + 1                       # >= 0
```

**Checklist:**
- [ ] Vòng ngoài `l < r` (1 ký tự không xóa được)?
- [ ] Vòng trong `l <= r` (khối giữa dài ≥ 2 xóa hết được)?
- [ ] Ăn **trọn khối**, không chỉ 1 ký tự mỗi đầu?
- [ ] Đã thử `"a"`, `"aa"`, `"aaa"`, `"aba"`, `"abba"`?
- [ ] Không cắt chuỗi trong vòng lặp (giữ `O(n)`).